In [13]:
import pandas as pd
import numpy as np

# UPDATE THIS PATH to match your SSD folder structure
train_path = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\UNSW_NB15_training-set.csv"
test_path  = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\UNSW_NB15_testing-set.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (82332, 45)
Test shape: (175341, 45)


In [14]:
# Quick sanity check: look at the label distribution in each file
print("=== train_df label counts ===")
print(train_df['label'].value_counts())

print("\n=== test_df label counts ===")
print(test_df['label'].value_counts())

=== train_df label counts ===
label
1    45332
0    37000
Name: count, dtype: int64

=== test_df label counts ===
label
1    119341
0     56000
Name: count, dtype: int64


In [15]:
# The Kaggle files are inversely labelled — swap to match official UNSW-NB15 convention
train_df, test_df = test_df, train_df

print("Swapped. New shapes:")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Swapped. New shapes:
Train shape: (175341, 45)
Test shape: (82332, 45)


In [16]:
# See all column names
print(train_df.columns.tolist())

# Look at the multi-class attack category breakdown
print("\nAttack category distribution:")
print(train_df['attack_cat'].value_counts())

['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Attack category distribution:
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64


In [17]:
# Drop the id column - it's just a row identifier, not a real feature
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])

# Check missing values and duplicates
print("Missing values in train set:", train_df.isnull().sum().sum())
print("Missing values in test set:", test_df.isnull().sum().sum())
print("Duplicate rows in train set:", train_df.duplicated().sum())
print("Duplicate rows in test set:", test_df.duplicated().sum())

Missing values in train set: 0
Missing values in test set: 0
Duplicate rows in train set: 67601
Duplicate rows in test set: 26387


In [18]:
# Remove duplicate rows (standard practice for UNSW-NB15, well documented in literature)
before_train, before_test = train_df.shape[0], test_df.shape[0]

train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()

print(f"Train: removed {before_train - train_df.shape[0]} duplicates -> new shape {train_df.shape}")
print(f"Test: removed {before_test - test_df.shape[0]} duplicates -> new shape {test_df.shape}")


Train: removed 67601 duplicates -> new shape (107740, 44)
Test: removed 26387 duplicates -> new shape (55945, 44)


In [19]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['proto', 'service', 'state']

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    encoders[col] = le

print("Training set categorical columns encoded successfully!")
train_df[categorical_cols].head()

Training set categorical columns encoded successfully!


,proto,service,state
0,113,0,2
1,113,0,2
2,113,0,2
3,113,3,2
4,113,0,2


In [20]:
# Build clean mapping dictionaries from training encoders
category_maps = {}
for col in categorical_cols:
    classes = list(encoders[col].classes_)
    category_maps[col] = {cat: idx for idx, cat in enumerate(classes)}

# Apply to test set, assigning a new shared code to anything unseen
for col in categorical_cols:
    mapping = category_maps[col]
    unseen_values = test_df.loc[~test_df[col].isin(mapping.keys()), col].unique()
    if len(unseen_values) > 0:
        print(f"{col}: unseen categories found -> {list(unseen_values)}")
    unseen_code = len(mapping)
    test_df[col] = test_df[col].apply(lambda x: mapping.get(x, unseen_code))

print("Test set categorical columns encoded successfully!")
test_df[categorical_cols].head()

state: unseen categories found -> ['ACC', 'CLO']
Test set categorical columns encoded successfully!


,proto,service,state
0,119,0,3
1,119,0,3
2,119,0,3
3,119,0,3
4,119,0,3


In [21]:
# Diagnostic check
print("Number of classes learned for 'proto':", len(encoders['proto'].classes_))
print("Sample of learned proto classes:", list(encoders['proto'].classes_)[:10])

print("\nSample of raw test_df proto values:", test_df['proto'].unique()[:10])


Number of classes learned for 'proto': 133
Sample of learned proto classes: ['3pc', 'a/n', 'aes-sp3-d', 'any', 'argus', 'aris', 'arp', 'ax.25', 'bbn-rcc', 'bna']

Sample of raw test_df proto values: [119   6 113  42  78  97  32  30  45  50]


In [22]:
from sklearn.preprocessing import StandardScaler

# Select numeric feature columns (exclude label and attack_cat)
feature_cols = [col for col in train_df.columns if col not in ['label', 'attack_cat']]

scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print("Features scaled successfully!")
train_df[feature_cols].head()

Features scaled successfully!


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports
0,-0.223580,0.087109,-0.679472,0.022178,-0.139493,-0.190846,-0.060940,-0.131903,-0.331102,1.010875,...,-0.520096,-0.293919,-0.210847,-0.491429,-0.130326,-0.130326,-0.314421,-0.582585,-0.610714,-0.071304
1,-0.127877,0.087109,-0.679472,0.022178,-0.093361,0.058156,-0.058802,0.102349,-0.331064,-0.752046,...,-0.520096,-0.293919,-0.210847,-0.328989,-0.130326,-0.130326,-0.314421,-0.582585,0.196045,-0.071304
2,0.048383,0.087109,-0.679472,0.022178,-0.127960,-0.102963,-0.060464,-0.059044,-0.331616,-0.752046,...,-0.272040,-0.293919,-0.210847,-0.166549,-0.130326,-0.130326,-0.314421,-0.382076,0.196045,-0.071304
3,0.058980,0.087109,0.403627,0.022178,-0.104894,-0.132257,-0.059279,-0.128555,-0.331620,-0.752046,...,-0.272040,-0.293919,-0.210847,-0.166549,7.398544,7.398544,-0.314421,-0.382076,-0.610714,-0.071304
4,-0.164180,0.087109,-0.679472,0.022178,-0.116427,-0.176199,-0.059701,-0.131365,-0.331451,1.029432,...,-0.272040,0.004774,-0.210847,5.843741,-0.130326,-0.130326,-0.314421,-0.382076,5.520653,-0.071304


In [23]:
import os

processed_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed"
os.makedirs(processed_folder, exist_ok=True)

train_save_path = os.path.join(processed_folder, "UNSWNB15_train_cleaned.csv")
test_save_path  = os.path.join(processed_folder, "UNSWNB15_test_cleaned.csv")

train_df.to_csv(train_save_path, index=False)
test_df.to_csv(test_save_path, index=False)

print("Cleaned data saved successfully!")
print("Train saved to:", train_save_path)
print("Test saved to:", test_save_path)

Cleaned data saved successfully!
Train saved to: C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_train_cleaned.csv
Test saved to: C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_test_cleaned.csv


In [24]:
print(y_train.unique())
print(y_train.name)  # confirms the column name too

NameError: name 'y_train' is not defined

In [ ]:
import pandas as pd

df = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_train_cleaned.csv")
print(df.columns.tolist())

In [ ]:
print(df['label'].unique())
print(df['label'].value_counts())

In [ ]:
# ============================================
# PHASE 1 — UNSW-NB15 — Model Training & Evaluation
# Mirrors 04_Phase1_NSLKDD_ModelTraining.ipynb
# ============================================

import pandas as pd
import numpy as np
import time
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, matthews_corrcoef, confusion_matrix)
from scipy.stats import wilcoxon
from itertools import combinations

# ---- 1. LOAD PROCESSED DATA ----
train_df = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_train_cleaned.csv")

# Drop attack_cat to prevent data leakage (it directly encodes the label)
X_train = train_df.drop(columns=['label', 'attack_cat'])
y_train = train_df['label']

# ---- 2. CLASS LABELS ----
POS_LABEL = 1                 # attack
LABELS = [0, 1]               # order: [normal, attack] for confusion_matrix

# ---- 3. MODELS ----
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
print("Cross-validation set up: 10 folds")

# ---- 4. FOLD LOOP ----
results = {name: {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'fpr': [], 'mcc': []} for name in models}

X_train_array = X_train.values
y_train_array = y_train.values

for fold_num, (train_idx, val_idx) in enumerate(skf.split(X_train_array, y_train_array), 1):
    X_fold_train, X_fold_val = X_train_array[train_idx], X_train_array[val_idx]
    y_fold_train, y_fold_val = y_train_array[train_idx], y_train_array[val_idx]

    print(f"\n--- Fold {fold_num}/10 ---")

    for name, model in models.items():
        start = time.time()
        model.fit(X_fold_train, y_fold_train)
        y_pred = model.predict(X_fold_val)
        elapsed = time.time() - start

        acc = accuracy_score(y_fold_val, y_pred)
        prec = precision_score(y_fold_val, y_pred, pos_label=POS_LABEL)
        rec = recall_score(y_fold_val, y_pred, pos_label=POS_LABEL)
        f1 = f1_score(y_fold_val, y_pred, pos_label=POS_LABEL)
        mcc = matthews_corrcoef(y_fold_val, y_pred)

        tn, fp, fn, tp = confusion_matrix(y_fold_val, y_pred, labels=LABELS).ravel()
        fpr = fp / (fp + tn)

        results[name]['accuracy'].append(acc)
        results[name]['precision'].append(prec)
        results[name]['recall'].append(rec)
        results[name]['f1'].append(f1)
        results[name]['fpr'].append(fpr)
        results[name]['mcc'].append(mcc)

        print(f"{name}: acc={acc:.4f}, f1={f1:.4f}, mcc={mcc:.4f} ({elapsed:.1f}s)")

print("\nAll folds complete!")

# ---- 5. SUMMARY TABLE ----
summary = pd.DataFrame({
    name: {metric: sum(vals)/len(vals) for metric, vals in results[name].items()}
    for name in results
}).T
summary = summary.round(4)
print(summary)

# ---- 6. WILCOXON SIGNED-RANK TESTS ----
model_names = list(results.keys())
print("\nWilcoxon signed-rank test results (F1 score, pairwise):\n")

for model_a, model_b in combinations(model_names, 2):
    scores_a = results[model_a]['f1']
    scores_b = results[model_b]['f1']
    stat, p_value = wilcoxon(scores_a, scores_b)
    significant = "SIGNIFICANT (p < 0.05)" if p_value < 0.05 else "not significant"
    print(f"{model_a} vs {model_b}: p={p_value:.4f} -> {significant}")

# ---- 7. SAVE RESULTS ----
results_folder = r"E:\School coursework\Dissertation\Datasets\UNSW-NB15\results"
os.makedirs(results_folder, exist_ok=True)

summary.to_csv(os.path.join(results_folder, "Phase1_UNSWNB15_summary.csv"))

fold_results = pd.DataFrame({
    (name, metric): vals
    for name in results
    for metric, vals in results[name].items()
})
fold_results.to_csv(os.path.join(results_folder, "Phase1_UNSWNB15_foldlevel.csv"))

print("Results saved successfully!")
print("Summary:", os.path.join(results_folder, "Phase1_UNSWNB15_summary.csv"))
print("Fold-level:", os.path.join(results_folder, "Phase1_UNSWNB15_foldlevel.csv"))